# Basline classifier using 1D CNN for time series data

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
import copy
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
MODEL_DIR = PROJECT_ROOT / "models"
data = np.load(PROCESSED_DIR / 'splits.npz')

X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']

print('Train:', X_train.shape, 'labels:', y_train.shape)
print('Val  :', X_val.shape)
print('Test :', X_test.shape)

Train: (14525, 6, 300) labels: (14525,)
Val  : (2075, 6, 300)
Test : (4150, 6, 300)


In [3]:
class HARDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


def make_loader(X, y=None, batch_size=128, shuffle=False):
    ds = HARDataset(X, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader   = make_loader(X_val, y_val, batch_size=128)
test_loader  = make_loader(X_test, y_test, batch_size=128)

In [4]:
X, y = next(iter(train_loader))

print(f"X shape: {X.shape}\ny shape: {y.shape}")
print("y unique:", torch.unique(y))
print(f"X mean: {X.mean():.4f}\nX std: {X.std():.4f}")
print(f"X_train mean: {X_train.mean():.4f}\nX_train std: {X_train.std():.4f}")

X shape: torch.Size([128, 6, 300])
y shape: torch.Size([128])
y unique: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])
X mean: 0.0053
X std: 0.4318
X_train mean: -0.0000
X_train std: 1.0000


In [5]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=18):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(3),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, device="cpu"):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                preds = out.argmax(dim=1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        acc = correct / total
        avg_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model

In [6]:
device="cpu"
model = BaselineCNN()
model = train_classifier(
    model,
    train_loader,
    val_loader,
    epochs=15,
    lr=5e-4,
    device=device
)
torch.save(model.state_dict(), MODEL_DIR / 'baseline_classifier.pth')

Epoch 1 | Loss: 2.3727 | Val Acc: 0.1089
Epoch 2 | Loss: 2.0375 | Val Acc: 0.0670
Epoch 3 | Loss: 1.8224 | Val Acc: 0.0665
Epoch 4 | Loss: 1.7454 | Val Acc: 0.0877
Epoch 5 | Loss: 1.7178 | Val Acc: 0.0882
Epoch 6 | Loss: 1.7588 | Val Acc: 0.0887
Epoch 7 | Loss: 1.6208 | Val Acc: 0.0887
Epoch 8 | Loss: 1.6837 | Val Acc: 0.0882
Epoch 9 | Loss: 1.5754 | Val Acc: 0.0887
Epoch 10 | Loss: 1.5076 | Val Acc: 0.0882
Epoch 11 | Loss: 1.6369 | Val Acc: 0.0887
Epoch 12 | Loss: 1.5148 | Val Acc: 0.0684
Epoch 13 | Loss: 1.5217 | Val Acc: 0.0867
Epoch 14 | Loss: 1.5904 | Val Acc: 0.0867
Epoch 15 | Loss: 1.5914 | Val Acc: 0.0877
